# core

> bioMONAI core


In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from torch import Tensor as torchTensor
from torch import tensor
from monai.data import MetaTensor

import torch.nn.functional as F

from random import randint

from skimage import util
from skimage.data import cells3d

In [ ]:
#| export
from torch import squeeze as torchsqueeze, max as torchmax, from_numpy as torch_from_numpy, device as torch_device
from torch.cuda import is_available as is_cuda_available

In [ ]:
#| export
from collections.abc import MutableSequence
from typing import MutableSequence
    
from fastai.callback.core import Callback
from fastai.data.all import DataLoaders, Path, trainable_params, delegates, hasattrs, Path, List, L, typedispatch
from fastai.optimizer import Adam, OptimWrapper, Optimizer
from fastai.vision.all import BypassNewMeta, DisplayedTransform, store_attr, DataBlock, Learner, ShowGraphCallback, CSVLogger, Any, minimum, steep, valley, slide


In [ ]:
show_doc(DataBlock)


---

[source](https://github.com/fastai/fastai/blob/master/fastai/callback/progress.py#LNone){target="_blank" style="float:right; font-size:smaller"}

### CSVLogger

>      CSVLogger (fname='history.csv', append=False)

Log the results displayed in `learn.path/fname`

In [ ]:
show_doc(DataLoaders)


In [ ]:
show_doc(Learner)


In [ ]:
show_doc(ShowGraphCallback)


In [ ]:
show_doc(CSVLogger)

## Engine

In [ ]:
#| export
class fastTrainer(Learner):
    """
    A custom implementation of the FastAI Learner class for training models in bioinformatics applications.
    
    Parameters:
        dataloaders (DataLoaders): The DataLoader objects containing training and validation datasets.
        model (callable): A callable model that will be trained on the dataset.
        loss_fn (Any | None): The loss function to optimize during training. If None, defaults to a suitable default.
        optimizer (Optimizer | OptimWrapper): The optimizer function to use. Defaults to Adam if not specified.
        lr (float | slice): Learning rate for the optimizer. Can be a float or a slice object for learning rate scheduling.
        splitter (callable): A callable that determines which parameters of the model should be updated during training.
        callbacks (Callback | MutableSequence | None): Optional list of callback functions to customize training behavior.
        metrics (Any | MutableSequence | None): Metrics to evaluate the performance of the model during training.
        csv_log (bool): Whether to log training history to a CSV file. If True, logs will be appended to 'history.csv'.
        path (str | Path | None): The base directory where models are saved or loaded from. Defaults to None.
        model_dir (str | Path): Subdirectory within the base path where trained models are stored. Default is 'models'.
        wd (float | int | None): Weight decay factor for optimization. Defaults to None.
        wd_bn_bias (bool): Whether to apply weight decay to batch normalization and bias parameters.
        train_bn (bool): Whether to update the batch normalization statistics during training.
        moms (tuple): Tuple of tuples representing the momentum values for different layers in the model. Defaults to FastAI's default settings if not specified.
        default_cbs (bool): Automatically include default callbacks such as ShowGraphCallback and CSVLogger.
    """
    
    def __init__(self, 
                 dataloaders: DataLoaders, 
                 model: callable, 
                 loss_fn: Any | None = None, 
                 optimizer: Optimizer | OptimWrapper = Adam, 
                 lr: float | slice = 1e-3, 
                 splitter: callable = trainable_params, 
                 callbacks: Callback | MutableSequence | None = None, 
                 metrics: Any | MutableSequence | None = None, 
                 csv_log: bool = False, 
                 show_graph: bool = True,
                 show_summary: bool = False,
                 find_lr: bool = False,
                 find_lr_fn = valley,
                 path: str | Path | None = None, 
                 model_dir: str | Path = 'models', 
                 wd: float | int | None = None, 
                 wd_bn_bias: bool = False, 
                 train_bn: bool = True, 
                 moms: tuple = ..., 
                 default_cbs: bool = True):
        cbs = callbacks if callbacks is not None else []  # Ensure cbs is a list
        if default_cbs:
            if show_graph:
                cbs.append(ShowGraphCallback())
            if csv_log:
                cbs.append(CSVLogger(fname='history.csv', append=False))
        
        super().__init__(dataloaders, model, loss_fn, optimizer, lr, splitter, cbs, metrics, path, model_dir, wd, wd_bn_bias, train_bn, moms)
        
        if show_summary:
                print(self.summary())
        if find_lr:
                self.lr_find(suggest_funcs=find_lr_fn)
                lr = float('%.1g'%(lr))
                print('Inferred learning rate: ', lr)


## Utils


In [ ]:
#| export
# maybe this should be changed for fastai store_attr()
def attributesFromDict(d):
    self = d.pop('self')
    for n, v in d.items():
        setattr(self, n, v)

In [ ]:
#| export
def get_device():
    return torch_device("cuda" if is_cuda_available() else "cpu")

In [ ]:
#| export
def img2float(image, force_copy=False):
    return util.img_as_float(image, force_copy=force_copy)

In [ ]:
#| export
def img2Tensor(image):
    return torchTensor(img2float(image))

---

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()